# ML Network Intrusion Detector
**IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares**
**Author:** Yashica Sharma

This project trains a machine learning model (scikit-learn) to look at network connection records and decide if each one is **normal** traffic or an **attack**. It uses the public **NSL-KDD** dataset, which is a well known benchmark for intrusion detection.

Basic flow: load data → look at it → clean and encode it → train models → check how good they are → test on a single connection.

## Step 1: Import the libraries

In [ ]:
# just the usual stuff, pandas for tables, sklearn for the ML part, matplotlib for graphs
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")  # hides the annoying warnings so output stays clean

## Step 2: Load the NSL-KDD dataset
The dataset comes as two text files with no header row: `KDDTrain+.txt` (training) and `KDDTest+.txt` (testing). If the files are sitting next to this notebook it uses those, otherwise it just downloads them from a public GitHub mirror, so it works straight away in Google Colab.

In [ ]:
# the files dont have column names so we have to give them ourselves
# 41 features + the label + a "difficulty" number at the end
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty"
]

base_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"

def load_file(name):
    # use the local copy if we have one, if not grab it from github
    path = name if os.path.exists(name) else base_url + name
    print("Loading", path)
    return pd.read_csv(path, names=columns)

train_df = load_file("KDDTrain+.txt")
test_df = load_file("KDDTest+.txt")

print("Training rows:", train_df.shape[0])
print("Testing rows:", test_df.shape[0])
train_df.head()

## Step 3: Explore the data a bit

In [ ]:
# quick look at what kind of traffic is in there
print(train_df["protocol_type"].value_counts())
print()
print("Top 10 labels in training data:")
print(train_df["label"].value_counts().head(10))

In [ ]:
# turning every attack name into just "attack" so its a simple yes/no problem
train_df["is_attack"] = (train_df["label"] != "normal").astype(int)
test_df["is_attack"] = (test_df["label"] != "normal").astype(int)

# bar chart so we can see if normal vs attack is roughly balanced
counts = train_df["is_attack"].value_counts().sort_index()
plt.figure(figsize=(5, 4))
plt.bar(["Normal", "Attack"], counts.values, color=["#4C9BE8", "#E8684C"])
plt.title("Normal vs Attack (training data)")
plt.ylabel("Number of connections")
plt.show()

## Step 4: Prepare the data
Three columns are text (`protocol_type`, `service`, `flag`), and ML models only understand numbers, so they get one-hot encoded (each value becomes its own 0/1 column). Then all the numbers are scaled so big columns like `src_bytes` dont drown out the small ones.

In [ ]:
# dropping stuff the model shouldnt see (the answer + the difficulty score)
drop_cols = ["label", "difficulty", "is_attack"]
X_train_raw = train_df.drop(columns=drop_cols)
X_test_raw = test_df.drop(columns=drop_cols)
y_train = train_df["is_attack"]
y_test = test_df["is_attack"]

# one-hot encode train and test together, so both end up with the exact same columns
# (the test set has a few services the train set doesnt have, this sorts that out)
combined = pd.concat([X_train_raw, X_test_raw], keys=["train", "test"])
combined = pd.get_dummies(combined, columns=["protocol_type", "service", "flag"])
X_train = combined.xs("train")
X_test = combined.xs("test")

# scaling, fit only on training data so we dont "peek" at the test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features after encoding:", X_train.shape[1])

## Step 5: Train the models
Two models are tried: a **Decision Tree** (simple, easy to explain) and a **Random Forest** (lots of trees voting together, usually more accurate).

In [ ]:
# decision tree first, max_depth stops it from memorising everything
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

# random forest, 100 trees, n_jobs=-1 just means use all cpu cores
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print("Decision Tree accuracy: {:.2f}%".format(accuracy_score(y_test, dt_pred) * 100))
print("Random Forest accuracy: {:.2f}%".format(accuracy_score(y_test, rf_pred) * 100))

## Step 6: Evaluate the results

In [ ]:
# precision = when it says attack, how often its right
# recall = out of all real attacks, how many it actually caught (this one matters most for security)
print("=== Decision Tree ===")
print(classification_report(y_test, dt_pred, target_names=["Normal", "Attack"]))
print("=== Random Forest ===")
print(classification_report(y_test, rf_pred, target_names=["Normal", "Attack"]))

In [ ]:
# confusion matrices side by side, shows exactly where each model messes up
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred, name in zip(axes, [dt_pred, rf_pred], ["Decision Tree", "Random Forest"]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Normal", "Attack"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# which features did the random forest care about most? top 10
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top10 = importances.sort_values(ascending=True).tail(10)

plt.figure(figsize=(7, 5))
plt.barh(top10.index, top10.values, color="#4C9BE8")
plt.title("Top 10 most important features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Step 7: Try it on a single connection

In [ ]:
# little helper that takes one row and says if its an attack or not
def check_connection(row_index):
    row = X_test.iloc[[row_index]]
    row_scaled = scaler.transform(row)
    guess = rf_model.predict(row_scaled)[0]
    real = test_df["label"].iloc[row_index]
    print("Connection #{} -> model says: {} | real label: {}".format(
        row_index, "ATTACK" if guess == 1 else "Normal", real))

# just testing a few random ones
for i in [0, 10, 100, 1000, 5000]:
    check_connection(i)

## Conclusion
The project shows that a simple scikit-learn model can learn the difference between normal network traffic and attacks just from connection details like bytes sent, error rates and the service used. Random Forest usually does better than a single Decision Tree. The test set (KDDTest+) also has some attack types that never appear in training, so the score is a bit lower than you'd get on data the model has seen before, which is actually a realistic picture of how a real IDS would struggle with brand new attacks.

**Possible improvements:** classify the attack type (DoS, Probe, R2L, U2R) instead of just yes/no, try other models like XGBoost, and handle the class imbalance for the rare attack types.